# 🛡️ SPECTRE_GRID — Retreinamento com NF-UQ-NIDS-v2

Este notebook foi gerado automaticamente para rodar no Google Colab.
Ele realiza as seguintes etapas:
1. Instala as dependências (PyTorch Geometric, etc).
2. Configura a sua chave do Kaggle.
3. Baixa o dataset `NF-UQ-NIDS-v2` (NetFlow v9).
4. Clona o seu repositório `ids-cnn-lstm-gnn`.
5. Roda o pré-processamento e o treinamento da STGNN na GPU (T4).

In [ ]:
!pip install torch-geometric pandas numpy scikit-learn
!pip install kaggle

## 🔑 Upload do Kaggle API Token
Para baixar o dataset de 2.1GB rapidamente, precisamos da sua chave do Kaggle.
Vá em [Kaggle > Account](https://www.kaggle.com/settings) e clique em **Create New Token** para baixar o `kaggle.json`.
Execute a célula abaixo e faça o upload desse arquivo.

In [ ]:
from google.colab import files
import os

# Pede para o usuário fazer o upload do arquivo
uploaded = files.upload()

if 'kaggle.json' in uploaded:
    !mkdir -p ~/.kaggle
    !mv kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json
    print("Kaggle token configurado com sucesso!")
else:
    print("ATENÇÃO: Você precisa fazer o upload do arquivo 'kaggle.json'.")

## 📦 Download do Dataset e Código

In [ ]:
import os

# 1. Clona o repositório
if not os.path.exists('ids-cnn-lstm-gnn'):
    !git clone https://github.com/abraaoteixeira/ids-cnn-lstm-gnn.git

# 2. Cria as pastas de dados
!mkdir -p ids-cnn-lstm-gnn/data/raw/benchmarks

# 3. Baixa e extrai o NF-UQ-NIDS-v2 (Kaggle)
os.chdir('ids-cnn-lstm-gnn/data/raw/benchmarks')
print("Baixando NF-UQ-NIDS-v2 do Kaggle (pode levar alguns minutos)...")
!kaggle datasets download -d mohanad-sarhan/nf-uq-nids-v2 --unzip

# Volta para a raiz do repositório
os.chdir('/content/ids-cnn-lstm-gnn')
print("Pronto para treinar!")

## 🧠 Execução do Treinamento

In [ ]:
import os
import glob
import pandas as pd

csv_files = glob.glob('data/raw/benchmarks/*.csv')
if not csv_files:
    print("Erro: Arquivos CSV não encontrados!")
else:
    # Pega o primeiro CSV baixado (geralmente NF-UQ-NIDS-v2.csv)
    csv_path = csv_files[0]
    print(f"Processando: {csv_path}")
    
    # Vamos pegar apenas os primeiros 200 mil registros para agilizar a prova de conceito
    # No retreinamento final, você pode processar o CSV inteiro
    df = pd.read_csv(csv_path, nrows=200000)
    df.to_csv('data/raw/benchmarks/NF-UQ-NIDS-v2-sample.csv', index=False)
    
    print("Rodando Preprocessor...")
    !python preprocessor.py --input_csv data/raw/benchmarks/NF-UQ-NIDS-v2-sample.csv --target_col Label
    
    print("\nRodando Training na GPU...")
    !python train.py --epochs 15 --batch_size 256

In [ ]:
from google.colab import files

# Baixa o modelo retreinado de volta para o seu PC
if os.path.exists('spectre_model_scripted.pt'):
    print("Baixando o modelo retreinado...")
    files.download('spectre_model_scripted.pt')
else:
    print("Modelo não encontrado. Ocorreu algum erro no treinamento?")